In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder

sns.set_style("whitegrid")

In [ ]:
DATASET_PATH = 'train.csv'
DATASET_PATH_TEST = 'test.csv'
TARGET = 'Price'
MAX_CATEGORIES = 20
DEFAULT_TOP_QUANTILE = 0.97

In [ ]:
df = pd.read_csv(DATASET_PATH)
df.head(10)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()



In [ ]:
df.isna().mean()


In [ ]:
df.isna().sum().sort_values(ascending=False)


In [ ]:
df.nunique().sort_values()

In [ ]:
df.duplicated().sum()
df[df.duplicated()]

In [ ]:
numeric_cols = df.select_dtypes(include=["number"]).columns

categorical_cols = df.select_dtypes(
    include=["object", "category", "bool"]
).columns

numeric_cols = numeric_cols.drop(["Id", "Helthcare_2", "Rooms", "Shops_1", ])
categorical_cols = categorical_cols.append(
    pd.Index(["Helthcare_2", "Rooms", "Shops_1"])
)


print(f"Числовые колонки ({len(numeric_cols)}):")
print(list(numeric_cols))

print(f"\nКатегориальные колонки ({len(categorical_cols)}):")
print(list(categorical_cols))


In [ ]:
def plot_numeric_distributions_with_quantile(df, numeric_cols):
    for col in numeric_cols:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        sns.histplot(df[col].dropna(), bins=100, kde=True, ax=axes[0])
        axes[0].set_title(f"{col} (с выбросами/экстремумами)")

        q_high = df[col].quantile(0.97)
        sns.histplot(
            df[col][df[col] <= q_high].dropna(),
            bins=100,
            kde=True,
            ax=axes[1]
        )
        axes[1].set_title(f"{col} (<= 97 перцентиль)")

        plt.tight_layout()
        plt.show()


In [ ]:
def plot_numeric_distributions(df, numeric_cols):
    for col in numeric_cols:
        plt.figure(figsize=(6, 4))

        sns.histplot(df[col].dropna(), bins=100, kde=True)
        plt.title(f"{col} (с выбросами/экстремумами)")
        plt.xlabel(col)
        plt.ylabel("Count")

        plt.tight_layout()
        plt.show()


In [ ]:
plot_numeric_distributions_with_quantile(df, numeric_cols)

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(6, 2))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot: {col}")
    plt.tight_layout()
    plt.show()


In [ ]:


for col in categorical_cols:
    vc = df[col].value_counts(dropna=False)

    if vc.shape[0] > MAX_CATEGORIES:
        print(f"{col}: слишком много категорий ({vc.shape[0]})")
        continue

    plt.figure(figsize=(6, 4))
    vc.plot(kind="bar")
    plt.title(f"Value counts: {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


In [ ]:
categorical_cols = categorical_cols.drop(["Ecology_2", "Ecology_3"])

In [ ]:
na_ratio = df.isna().mean().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
na_ratio.plot(kind="bar")
plt.title("Missing values ratio per column")
plt.ylabel("Fraction of NaN")
plt.tight_layout()
plt.show()


In [ ]:
def fill_numeric_with_median(df, numeric_cols):
    df = df.copy()
    medians = {}

    for col in numeric_cols:
        if col not in df.columns:
            continue

        median_value = df[col].median()
        df[col] = df[col].fillna(median_value)
        medians[col] = median_value

    return df, medians


In [ ]:
df, medians = fill_numeric_with_median(
    df,
    ["Healthcare_1"]
)

In [ ]:
# считает среднее соотношение Общей площади и жилой, затем заполняет пропуски в LifeSquare

def fill_lifesquare_from_square(
    df,
    square_col="Square",
    lifesquare_col="LifeSquare",
    min_square=10,
):
    df = df.copy()

    mask_valid = (
        df[square_col].notna()
        & df[lifesquare_col].notna()
        & (df[square_col] > min_square)
    )

    ratios = df.loc[mask_valid, lifesquare_col] / df.loc[mask_valid, square_col]
    k = ratios.median()

    fill_mask = df[lifesquare_col].isna() & df[square_col].notna()
    df.loc[fill_mask, lifesquare_col] = df.loc[fill_mask, square_col] * k

    return df, k


In [ ]:
df, life_ratio = fill_lifesquare_from_square(df)

In [ ]:
na_ratio = df.isna().mean().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
na_ratio.plot(kind="bar")
plt.title("Missing values ratio per column")
plt.ylabel("Fraction of NaN")
plt.tight_layout()
plt.show()

In [ ]:
plot_numeric_distributions_with_quantile(df, numeric_cols)

In [ ]:
# давит экстремальные значения с двух сторон в указанных колонках к указанному перцентилю 
def clip_numeric_columns(df, numeric_cols, clip_rules):
    df = df.copy()
    clip_bounds = {}

    numeric_cols = [c for c in numeric_cols if c in df.columns]

    for col in numeric_cols:
        if col not in clip_rules:
            continue

        q_low, q_high = clip_rules[col]

        low = df[col].quantile(q_low)
        high = df[col].quantile(q_high)

        df[col] = df[col].clip(lower=low, upper=high)
        clip_bounds[col] = (low, high)

    return df, clip_bounds


In [ ]:
numeric_cols

In [ ]:


CLIP_RULES = {
    "Square": (0.01, DEFAULT_TOP_QUANTILE),
    "LifeSquare": (0.01, DEFAULT_TOP_QUANTILE),
    "KitchenSquare": (0.01, DEFAULT_TOP_QUANTILE),
    "HouseYear": (0.01, 0.99),
}

df, clip_bounds = clip_numeric_columns(df, numeric_cols, CLIP_RULES)

In [ ]:
plot_numeric_distributions(df, numeric_cols)

1) импутация. жилая площадь заполнена от общей площади, healthcare медианой
2) отрезали выбросы и края хвостов
3) пару колонок отбросили как констсантные

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
def select_numeric_and_categorical(df, numeric_cols, categorical_cols):
    numeric_cols = list(numeric_cols)
    categorical_cols = list(categorical_cols)

    cols_to_keep = [col for col in numeric_cols + categorical_cols if col in df.columns]
    return df[cols_to_keep].copy()



In [ ]:
numeric_cols

In [ ]:
categorical_cols

In [ ]:
df = select_numeric_and_categorical(
    df,
    numeric_cols,
    categorical_cols,
)

In [ ]:
df.head()

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

In [ ]:
# иначе catboost не понимает категориальные признаки
for col in categorical_cols:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)

In [ ]:
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=100,
)

model.fit(
    X_train,
    y_train,
    cat_features=list(categorical_cols),
    eval_set=(X_test, y_test),
    use_best_model=True,
)


In [ ]:
from sklearn.metrics import r2_score

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
print(f"R2 score: {r2:.4f}")


In [ ]:
import pandas as pd
from pandas import Index
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score
from typing import List, Optional

class RealEstatePipeline:
    """
    Пайплайн для предсказания цены недвижимости.
    Автоматически определяет числовые и категориальные признаки,
    приводит категориальные к строке (для CatBoost),
    обучает модель и может делать предсказания.
    """
    
    def __init__(
        self,
        catboost_params: dict = None,
        target: str = 'Price',
        id_col: str = 'Id'
    ):
        self.target = target
        self.id_col = id_col
        
        self.default_catboost_params = {
            'iterations': 1000,
            'depth': 6,
            'learning_rate': 0.05,
            'loss_function': 'RMSE',
            'random_seed': 42,
            'verbose': 100
        }
        
        # Если передали свои параметры — обновляем
        if catboost_params:
            self.default_catboost_params.update(catboost_params)
        
        self.model = None
        self.numeric_cols: List[str] = []
        self.categorical_cols: List[str] = []
        self.feature_columns: List[str] = []  # порядок признаков для модели


    def preprocess_dataframe(self, df: pd.DataFrame) -> tuple[pd.DataFrame, Index, Index]:
        numeric_cols = df.select_dtypes(include=["number"]).columns

        categorical_cols = df.select_dtypes(
            include=["object", "category", "bool"]
        ).columns

        # numeric_cols = numeric_cols.drop(["Id", "Helthcare_2", "Rooms", "Shops_1", ])
        numeric_cols = numeric_cols.drop(["Id", "Helthcare_2", "Rooms", "Shops_1", ])
        categorical_cols = categorical_cols.append(
            pd.Index(["Helthcare_2", "Rooms", "Shops_1"])
        )
        categorical_cols = categorical_cols.drop(["Ecology_2", "Ecology_3"])
        # plot_numeric_distributions_with_quantile(df, numeric_cols)
        # for col in numeric_cols:
        #     plt.figure(figsize=(6, 2))
        #     sns.boxplot(x=df[col])
        #     plt.title(f"Boxplot: {col}")
        #     plt.tight_layout()
        #     plt.show()

        MAX_CATEGORIES = 20

        # for col in categorical_cols:
        #     vc = df[col].value_counts(dropna=False)

        #     if vc.shape[0] > MAX_CATEGORIES:
        #         print(f"{col}: слишком много категорий ({vc.shape[0]}), пропускаем график")
        #         continue

        #     plt.figure(figsize=(6, 4))
        #     vc.plot(kind="bar")
        #     plt.title(f"Value counts: {col}")
        #     plt.xlabel(col)
        #     plt.ylabel("Count")
        #     plt.tight_layout()
        #     plt.show()
        
        na_ratio = df.isna().mean().sort_values(ascending=False)

        # plt.figure(figsize=(8, 4))
        # na_ratio.plot(kind="bar")
        # plt.title("Missing values ratio per column")
        # plt.ylabel("Fraction of NaN")
        # plt.tight_layout()
        # plt.show()

        df, medians = fill_numeric_with_median(df, ["Healthcare_1"])
        df, life_ratio = fill_lifesquare_from_square(df)
        # plot_numeric_distributions_with_quantile(df, numeric_cols)
        DEFAULT_TOP_QUANTILE = 0.97

        CLIP_RULES = {
            "Square": (0.01, DEFAULT_TOP_QUANTILE),
            "LifeSquare": (0.01, DEFAULT_TOP_QUANTILE),
            "KitchenSquare": (0.01, DEFAULT_TOP_QUANTILE),
            "HouseYear": (0.01, 0.99),
        }

        df, clip_bounds = clip_numeric_columns(df, numeric_cols, CLIP_RULES)
        # plot_numeric_distributions(df, numeric_cols)
        df = select_numeric_and_categorical(
            df,
            numeric_cols,
            categorical_cols,
        )

        self.categorical_cols = categorical_cols

        return df, numeric_cols, categorical_cols
    
    def prepare_train_test_split(
        self,
        df: pd.DataFrame,
        X_prepared: pd.DataFrame,
        categorical_cols: Index
    ) -> tuple:

        if self.target not in df.columns:
            raise ValueError(f"Таргет колонка '{self.target}' не найдена в данных")
        
        y = df[self.target]
        
        X = X_prepared.drop(columns=[self.target], errors='ignore')
        # X = X_prepared
        
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=42,
        )
        
        # Приводим категориальные к строке (для CatBoost)
        # for col in categorical_cols:
        #     if col in X_train.columns:
        #         X_train[col] = X_train[col].astype(str)
        #         X_test[col] = X_test[col].astype(str)
        
        print(f"Обучение модели на {len(X_train)} объектах, валидация на {len(X_test)}")
        
        return X_train, X_test, y_train, y_test
    
    def convert_categorical_to_str(self, categorical_cols, X_train, X_test):
        """Простое преобразование категориальных колонок в строки"""
        X_train = X_train.copy()
        X_test = X_test.copy()
        
        for col in categorical_cols:
            if col in X_train.columns:
                X_train[col] = X_train[col].astype(str)
            if col in X_test.columns:
                X_test[col] = X_test[col].astype(str)
        
        return X_train, X_test

    def fit(self, df_train: pd.DataFrame, val_size: float = 0.2, random_state: int = 42):
        """
        Обучает модель на полном датафрейме (с таргетом)
        """
        print("Подготовка обучающих данных...")
        X_prepared, _, categorical_cols = self.preprocess_dataframe(df_train)

        self.feature_columns = [col for col in X_prepared.columns if col not in [self.id_col, self.target]]

        X_train, X_test, y_train, y_test = self.prepare_train_test_split(df, X_prepared, categorical_cols)
        X_train, X_test = self.convert_categorical_to_str(categorical_cols, X_train, X_test)

        print('fit cat cols:', list(categorical_cols))
        print('fit all cols:', list(X_train.columns))
        
        self.model = CatBoostRegressor(**self.default_catboost_params)

        self.model.fit(
            X_train,
            y_train,
            cat_features=list(categorical_cols),
            # cat_features=list(cat_feature_indices),
            eval_set=(X_test, y_test),
            use_best_model=True,
        )
        
        
        # Оценка на валидации
        val_pred = self.model.predict(X_test)
        r2 = r2_score(y_test, val_pred)
        print(f"Качество на валидации: R² = {r2:.4f}")
        
        return self

    def _prepare_test_features(self, df_test: pd.DataFrame):
        """
        Готовит признаки для теста строго по структуре, запомненной на train
        """
        if self.feature_columns is None:
            raise ValueError("Сначала обучите модель на train (вызовите .fit())")

        if self.id_col not in df_test.columns:
            raise KeyError(f"Колонка '{self.id_col}' отсутствует после предобработки теста!")
        
        ids = df_test[self.id_col].astype(int)

        # Применяем ту же предобработку, что и на train
        X_prepared, _, categorical_cols = self.preprocess_dataframe(df_test)
        X_prepared, _ = self.convert_categorical_to_str(categorical_cols, X_prepared, X_prepared)

        print('_prepare_test_features cat cols:', list(categorical_cols))

        X = X_prepared

        print('_prepare_test_features all cols:', list(X.columns))

        print(f"Готово: X_test shape = {X.shape} (ожидаемо колонок: {len(self.feature_columns)})")
        return X, ids

    def predict(self, df_test: pd.DataFrame) -> pd.DataFrame:
        if self.model is None:
            raise ValueError("Модель не обучена! Сначала вызовите .fit()")
        
        print("=== Подготовка тестовых данных ===")
        X_test, ids = self._prepare_test_features(df_test)
        
        print(f"Предсказание на {len(X_test)} объектах...")
        predictions = self.model.predict(X_test)
        
        return pd.DataFrame({
            'Id': ids,
            'Price': predictions
        })

    def predict_submission(self, df_test: pd.DataFrame, filename: str = "SSHirkin_predictions.csv"):
        submission = self.predict(df_test)
        
        submission.to_csv(filename, index=False)
        print(f"Готово! Файл сохранён: {filename}")
        print(f"Строк в файле: {len(submission) + 1} (включая заголовок) → должно быть 5001")
        
        return submission

    def save_model(self, path: str = "realestate_catboost.cbm"):
        if self.model:
            self.model.save_model(path)
            print(f"Модель сохранена: {path}")

In [ ]:
# pipeline = RealEstatePipeline()

# df = pd.read_csv(DATASET_PATH)

# pipeline.fit(df)           # ← один датафрейм
# submission = pipeline.predict_submission(df_test)  # ← другой датафрейм

In [67]:
# Загрузка данных
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

# Создаём пайплайн
pipeline = RealEstatePipeline()

# ←←← ВАЖНО: вставь свой код предобработки в preprocess_dataframe!

# Обучаем
pipeline.fit(df_train)

# После fit
# print("Признаки, запомненные на train:")
# print(pipeline.feature_columns)

# Перед predict
df_test = pd.read_csv('test.csv')
# print("\nКолонки в исходном test.csv:")
# print(df_test.columns.tolist())

# X_prepared_test, _, _ = pipeline.preprocess_dataframe(df_test)
# print("\nКолонки после preprocess_dataframe на test:")
# print(X_prepared_test.columns.tolist())

# Предсказываем и сохраняем


Подготовка обучающих данных...
Обучение модели на 8000 объектах, валидация на 2000
fit cat cols: ['Shops_2', 'Helthcare_2', 'Rooms', 'Shops_1']
fit all cols: ['DistrictId', 'Square', 'LifeSquare', 'KitchenSquare', 'Floor', 'HouseFloor', 'HouseYear', 'Ecology_1', 'Social_1', 'Social_2', 'Social_3', 'Healthcare_1', 'Shops_2', 'Helthcare_2', 'Rooms', 'Shops_1']
0:	learn: 90210.4789597	test: 91267.0818285	best: 91267.0818285 (0)	total: 53ms	remaining: 53s
100:	learn: 46834.3901361	test: 51844.5822048	best: 51844.5822048 (100)	total: 3.92s	remaining: 34.9s
200:	learn: 43099.0517940	test: 49996.4901610	best: 49996.4901610 (200)	total: 6.69s	remaining: 26.6s
300:	learn: 40691.5923543	test: 48972.8686423	best: 48972.8686423 (300)	total: 9.62s	remaining: 22.3s
400:	learn: 38753.9918116	test: 48297.4752060	best: 48297.4752060 (400)	total: 12.7s	remaining: 18.9s
500:	learn: 37210.0378681	test: 48010.1813645	best: 48010.1813645 (500)	total: 15.5s	remaining: 15.4s
600:	learn: 35913.9974875	test: 47

In [68]:
pipeline.predict_submission(df_test, filename="SSHirkin_predictions.csv")

=== Подготовка тестовых данных ===
_prepare_test_features cat cols: ['Shops_2', 'Helthcare_2', 'Rooms', 'Shops_1']
_prepare_test_features all cols: ['DistrictId', 'Square', 'LifeSquare', 'KitchenSquare', 'Floor', 'HouseFloor', 'HouseYear', 'Ecology_1', 'Social_1', 'Social_2', 'Social_3', 'Healthcare_1', 'Shops_2', 'Helthcare_2', 'Rooms', 'Shops_1']
Готово: X_test shape = (5000, 16) (ожидаемо колонок: 16)
Предсказание на 5000 объектах...
Готово! Файл сохранён: SSHirkin_predictions.csv
Строк в файле: 5001 (включая заголовок) → должно быть 5001


,Id,Price
0,725,162275.868770
1,15856,215817.226576
2,5480,254263.134124
3,15664,366446.314728
4,14275,142162.048455
...,...,...
4995,8180,246634.088136
4996,4695,128294.660576
4997,5783,327580.716688
4998,4780,189234.014978
